In [0]:
dbutils.widgets.text("max_events", "300", "Max events to read")
dbutils.widgets.text("eventhub_name", "elina-sharabura-eventhub", "Target Event Hub name")

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("eventhub_producer")

In [0]:
connection_str = dbutils.secrets.get(
    scope="elina_sharabura-personal-scope",
    key="elina-sharabura-eventhub-connstr"
)

In [0]:
from azure.eventhub import EventHubProducerClient, EventData

EVENTHUB_NAME = dbutils.widgets.get("eventhub_name")

producer = EventHubProducerClient.from_connection_string(
    conn_str=connection_str,
    eventhub_name=EVENTHUB_NAME
)

In [0]:
import requests
import sseclient
import json

url = "https://stream.wikimedia.org/v2/stream/recentchange"
headers = {
    "Accept": "text/event-stream",
    "User-Agent": "DatabricksLab3-EventHubProducer/1.0 (elina.sharabura@softserve.academy)"
}
response = requests.get(url, stream=True, headers=headers)
client = sseclient.SSEClient(response)

In [0]:
MAX_EVENTS = int(dbutils.widgets.get("max_events"))
events_read = 0
events_sent = 0

batch = producer.create_batch()

for event in client.events():
    if not event.data:
        continue
    try:
        change = json.loads(event.data)
    except json.JSONDecodeError:
        continue

    payload = json.dumps(change)

    try:
        batch.add(EventData(payload))
    except ValueError:
        producer.send_batch(batch)
        events_sent += len(batch)
        batch = producer.create_batch()
        batch.add(EventData(payload))

    events_read += 1
    if events_read >= MAX_EVENTS:
        break

if len(batch) > 0:
    producer.send_batch(batch)
    events_sent += len(batch)

producer.close()
logger.info(f"Read {events_read} events, sent {events_sent} to {EVENTHUB_NAME}")